# Retrieval-Augmented Generation (RAG) from Scratch

Welcome to this interactive walkthrough of **Retrieval-Augmented Generation (RAG)**.

### What is RAG and Why Do We Need It?
Large Language Models (LLMs) are powerful, but they have two fundamental limitations:
1. **Knowledge Cutoffs**: They don't know about private data, internal documents, or events after their training date.
2. **Hallucination**: When unsure, LLMs can fabricate convincing but incorrect facts.

**RAG solves this by acting like an 'open-book exam' for the LLM:**
Instead of asking the LLM to recall facts from its parametric memory, we:
1. Retrieve the most relevant passages from our documents.
2. Inject those passages directly into the prompt as context.
3. Instruct the LLM to answer using *only* that context.

## Setup: Import Path & Dependencies

In [ ]:
import sys
from pathlib import Path

# Ensure project root and src/ are in Python path
ROOT = Path.cwd().parent if Path.cwd().name == 'experiment' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))

print("Project root configured at:", ROOT)

## Step 1: Document Ingestion
In RAG, raw unstructured data (text files, articles, PDFs) are loaded into structured `Document` objects.

In [ ]:
from rag_engine.documents import Document, load_text_file

# Load demo articles bundled in the project
demo_dir = ROOT / 'tests' / 'fixtures' / 'demo_corpus'
documents = [load_text_file(f) for f in sorted(demo_dir.glob('*.txt'))]

for doc in documents:
    print(f"Loaded '{doc.title}' ({len(doc.text.split())} words)")

## Step 2: Chunking Strategy
Why can't we just embed the entire document as one giant vector?
- Large documents dilute specific facts.
- Embedding models typically have token limits (e.g., 256–512 tokens).

By splitting text into **smaller chunks with overlap**, we ensure high precision retrieval without cutting important sentences in half.

In [ ]:
from rag_engine.chunking import chunk_documents

# Split documents into chunks of 80 words with 15-word overlap
chunks = chunk_documents(documents, chunk_size=80, chunk_overlap=15)
print(f"Total chunks generated: {len(chunks)}\n")

# Inspect the first chunk
sample_chunk = chunks[0]
print(f"Chunk ID: {sample_chunk.chunk_id}")
print(f"Document: {sample_chunk.title}")
print(f"Content:\n{sample_chunk.text}")

## Step 3: Embeddings & Vector Similarity Search
Embedding models convert text strings into numerical vectors in a semantic vector space.
We use cosine similarity to measure how semantically close a search query is to each chunk.

In [ ]:
from rag_engine.embeddings import HashEmbeddings, cosine_similarity
from rag_engine.vector_store import VectorStore

# Create an in-memory vector store and add our chunks
vector_store = VectorStore(embedding_model=HashEmbeddings())
vector_store.add_chunks(chunks)

query = "What temperature do qubits need to operate?"
vector_results = vector_store.search(query, top_k=2)

print(f"Query: '{query}'\n")
for idx, hit in enumerate(vector_results, start=1):
    print(f"[Rank {idx}] Score: {hit['score']} | Doc: {hit['chunk'].title}")
    print(f"Snippet: {hit['chunk'].text[:150]}...\n")

## Step 4: Hybrid Search (Dense Vectors + BM25 Keywords)
- **Dense Vector Search** captures conceptual similarity (e.g. 'space telescope' -> 'astronomy').
- **BM25 Search** captures exact keywords and acronyms (e.g. 'NIRCam', '15 millikelvin', 'L2').
- **Reciprocal Rank Fusion (RRF)** combines both rankings into a balanced result.

In [ ]:
from rag_engine.bm25_search import BM25SearchIndex
from rag_engine.hybrid_search import HybridRetriever

bm25_index = BM25SearchIndex()
bm25_index.build(chunks)

hybrid_retriever = HybridRetriever(vector_store, bm25_index)
retrieval_output = hybrid_retriever.retrieve("NIRCam gold-plated beryllium", top_k=2)

print("Top Hybrid Retrieval Results:")
for idx, item in enumerate(retrieval_output['results'], start=1):
    print(f"[Rank {idx}] Fused Score: {item['score']} | Source: {item['chunk'].title}")
    print(f"Text: {item['chunk'].text[:200]}...\n")

## Step 5: End-to-End Pipeline & Generation
Now we run the complete `RAGPipeline` class which packages all the steps into a clean interface:
Query -> Hybrid Retrieve -> Format Prompt Context -> Generate Answer -> Evaluate.

In [ ]:
from rag_engine.pipeline import RAGPipeline

pipeline = RAGPipeline()
pipeline.load_demo_corpus()

question = "Where is the James Webb Space Telescope stationed in space?"
response = pipeline.query(question)

print("=" * 60)
print("Question:", response['question'])
print("Answer:\n", response['answer'])
print("\nSources Cited:", response['sources'])
print("Evaluation Metrics:", response['evaluation'])
print(f"Latency: {response['latency_ms']} ms")
print("=" * 60)